# Приступаем к работе над моделью детектинга  

Возьмём предобученную модель Yolo (v10), размеченный датасет, предобработаем его и обучим.

In [1]:
import os
import cv2
import time

from ultralytics import YOLO
import shutil
import numpy as np

### Подключаем Torch (Cuda)  

Данная версия Torch поддерживает соединение с cuda 13.1

In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

import torchvision
print(torchvision.__version__) 

2.9.1+cu128
True
0.24.1+cu128


# Очистка неразмеченных данных  

In [3]:
from pathlib import Path

images_path = Path('D://AIM/AI-Flow-Detecting/ai-core/dataset/train')

for img_file in images_path.glob("*.jpg"):
    txt_file = img_file.with_suffix('.txt')
    
    # Проверка: нет файла .txt ИЛИ он пустой (включая только пробелы/переносы)
    if not txt_file.exists() or not txt_file.read_text(encoding='utf-8').strip():
        img_file.unlink()  # Удалить .jpg
        if txt_file.exists():
            txt_file.unlink()  # Опционально: удалить и пустой .txt
        print(f"Удалено изображение (и, возможно, пустой .txt): {img_file}")


In [4]:
# PATHS
RAW_DATASET = Path("dataset/train")
PREPROCESSED_DATASET = Path("dataset/train_preprocesse_ul")

PREPROCESSED_DATASET.mkdir(parents=True, exist_ok=True)

## Предобработка кадров датасета для обучения

In [5]:
def adaptive_gamma(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mean = gray.mean()

    if mean < 80:      # ночь
        gamma = 1.7
    elif mean < 115:   # сумерки
        gamma = 1.3
    else:              # день
        gamma = 1.0

    inv = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv) * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(img, table)


def clahe_luminance(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l = clahe.apply(l)

    lab = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


# def silhouette_boost(img):
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

#     grad_x = cv2.Sobel(gray, cv2.CV_16S, 1, 0)
#     grad_y = cv2.Sobel(gray, cv2.CV_16S, 0, 1)

#     abs_x = cv2.convertScaleAbs(grad_x)
#     abs_y = cv2.convertScaleAbs(grad_y)

#     edges = cv2.addWeighted(abs_x, 0.5, abs_y, 0.5, 0)
#     edges = cv2.normalize(edges, None, 0, 255, cv2.NORM_MINMAX)

#     mask = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
#     return cv2.addWeighted(img, 0.9, mask, 0.1, 0)

def silhouette_boost(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    grad_x = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)

    mag = cv2.magnitude(grad_x, grad_y)
    mag = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    mag = mag.astype(np.uint8)

    edges = cv2.cvtColor(mag, cv2.COLOR_GRAY2BGR)

    return cv2.addWeighted(img, 0.85, edges, 0.25, 0)

def mild_sharpen(img):
    kernel = np.array([[0, -0.5, 0],
                       [-0.5, 3, -0.5],
                       [0, -0.5, 0]])
    return cv2.filter2D(img, -1, kernel)

# Если люди мелкие
def silhouette_morph(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 60, 140)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    edges = cv2.dilate(edges, kernel, iterations=1)

    edges = cv2.GaussianBlur(edges, (3, 3), 0)
    edges = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

    return cv2.addWeighted(img, 0.85, edges, 0.3, 0)


def local_contrast(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mean = gray.mean()

    if mean > 120:  # дневной свет
        return img  # ❌ не трогаем

    blur = cv2.GaussianBlur(img, (0, 0), sigmaX=10)
    return cv2.addWeighted(img, 1.2, blur, -0.2, 0)

def full_preprocess(img):
    img = adaptive_gamma(img)
    img = clahe_luminance(img)
    img = local_contrast(img) 
    img = silhouette_boost(img)
    # img = silhouette_morph(img)
    img = mild_sharpen(img)
    return img

# =========================
# DATASET PREPROCESSING
# =========================

print("Предобработка датасета...")

for img_file in RAW_DATASET.glob("*.jpg"):
    txt_file = img_file.with_suffix(".txt")
    if not txt_file.exists():
        continue

    img = cv2.imread(str(img_file))
    if img is None:
        continue

    processed = full_preprocess(img)

    cv2.imwrite(str(PREPROCESSED_DATASET / img_file.name), processed)
    shutil.copy(txt_file, PREPROCESSED_DATASET / txt_file.name)

print("Предобработка завершена")

Предобработка датасета...
Предобработка завершена


## Распределение в датасете  

In [6]:
from sklearn.model_selection import train_test_split

dataset_path = "dataset/train_preprocesse_ul"
train_dest = "dataset/train_split_ul"
val_dest = "dataset/val_split_ul"

val_ratio = 0.2
random_seed = 42

Path(train_dest).mkdir(parents=True, exist_ok=True)
Path(val_dest).mkdir(parents=True, exist_ok=True)

all_files = [f for f in os.listdir(dataset_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
print(f"Найдено {len(all_files)} изображений")

train_files, val_files = train_test_split(
    all_files,
    test_size=val_ratio,
    random_state=random_seed
)

def copy_files(files, destination):
    for file in files:
        base = os.path.splitext(file)[0]
        shutil.copy(os.path.join(dataset_path, file), os.path.join(destination, file))
        txt = f"{base}.txt"
        src_txt = os.path.join(dataset_path, txt)
        if os.path.exists(src_txt):
            shutil.copy(src_txt, os.path.join(destination, txt))

copy_files(train_files, train_dest)
copy_files(val_files, val_dest)

print("Train / Val split готов")

Найдено 709 изображений
Train / Val split готов


## Обучение модели  

In [ ]:
import torch
from ultralytics import YOLO

MODEL_PATH = 'yolo11m.pt'
OUTPUT_NAME = 'detect_finetuned'
PROJECT_DIR = 'runs/train'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используется: {device}")

# Автоматическое включение AMP только на GPU
use_amp = True if device == 'cuda' else False

model = YOLO(MODEL_PATH).to(device)

model.train(
    data='data-ul.yaml',
    epochs=65,
    imgsz=640,
    batch=16,
    device=device,
    name=OUTPUT_NAME,
    project=PROJECT_DIR,

    augment=True,
    optimizer='AdamW',

    workers=4,
    amp=use_amp,

    hsv_h=0.03,
    hsv_s=0.7,
    hsv_v=0.8,

    translate=0.3,
    scale=0.5,
    fliplr=0.5,
    mosaic=0.8,
    mixup=0.1,
    degrees=10.0,

    lr0=3e-4,
    lrf=0.01,

    rect=False,
    cache='ram',
    plots=True,
    save=True,
)

Используется: cuda
Ultralytics 8.4.33  Python-3.12.7 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4060, 8187MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data-ul.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.03, hsv_s=0.7, hsv_v=0.8, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0003, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=0.8, multi_scale=0.0, name=detect_finetuned8, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, pa

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001EEAB990920>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

### Сохранение модели  

In [13]:
model.save('runs/result/preprocessdetect122spring.pt')
print("✅ Модель дообучена и сохранена.")

✅ Модель дообучена и сохранена.


In [9]:
metrics = model.val(
    data="data-ul.yaml",
    imgsz=640,
    conf=0.25
)

Ultralytics 8.4.33  Python-3.12.7 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4060, 8187MiB)
YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 160.759.1 MB/s, size: 2086.0 KB)
val: Scanning D:\FQWProject\AI-Flow-Detecting\ai-core\dataset\val_split_ul.cache... 142 images, 0 backgrounds, 5 corrupt: 100% ━━━━━━━━━━━━ 142/142  0.0s
val: D:\FQWProject\AI-Flow-Detecting\ai-core\dataset\val_split_ul\event_20260323_174143.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: D:\FQWProject\AI-Flow-Detecting\ai-core\dataset\val_split_ul\event_20260324_131641.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: D:\FQWProject\AI-Flow-Detecting\ai-core\dataset\val_split_ul\event_20260324_165834.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels 

# Зависимости для проверки  

In [10]:
import cv2
import numpy as np
import signal
from ultralytics import YOLO
from collections import defaultdict

import sys

In [11]:
# ================== GRACEFUL SHUTDOWN ==================
running = True

def signal_handler(sig, frame):
    global running
    print("\nОстановка...")
    running = False

signal.signal(signal.SIGINT, signal_handler)

# MODEL
print("Загрузка модели...")
model = YOLO("runs/result/preprocessdetect2spring.pt")

# VIDEO SOURCE
SOURCE = "https://video2.interra.ru/glaz.naroda.121-c9e033c05f/index.m3u8?token=3.9CzUU5u-AAAAAAAAAEsAAAAAAAAAAHTKPf595bILj3OTg5Az34JUkT1F"

cap = cv2.VideoCapture(SOURCE)
if not cap.isOpened():
    raise RuntimeError("Не удалось открыть видеопоток")

# ================== TRACK STORAGE ==================
tracks = defaultdict(list)
seen_track_ids = set()

print("▶️ Старт обработки. Нажми Q или Ctrl+C")
frame_id = 0

# ================== MAIN LOOP ==================
while running and cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Потеря кадра, повтор...")
        cv2.waitKey(500)
        continue

    frame_id += 1

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.4,
        iou=0.4,
        classes=[0],
        verbose=False
    )

    current_ids = set()

    if results and results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy()

        for box, track_id in zip(boxes, ids):
            track_id = int(track_id)
            current_ids.add(track_id)
            seen_track_ids.add(track_id)

            x1, y1, x2, y2 = map(int, box)
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            tracks[track_id].append((cx, cy))
            if len(tracks[track_id]) > 30:
                tracks[track_id].pop(0)

            # ANALYSIS
            state = "unknown"
            color = (255, 255, 0)
            speed = 0.0
            dx = dy = 0

            if len(tracks[track_id]) >= 2:
                x_prev, y_prev = tracks[track_id][-2]
                dx = cx - x_prev
                dy = cy - y_prev
                speed = np.sqrt(dx*dx + dy*dy)

                if speed < 2:
                    state = "standing"
                    color = (0, 0, 255)
                else:
                    state = "moving"
                    color = (0, 255, 0)

            # ================== DRAW TRAJECTORY ==================
            for i in range(1, len(tracks[track_id])):
                cv2.line(
                    frame,
                    tracks[track_id][i-1],
                    tracks[track_id][i],
                    (100, 100, 255),
                    2
                )

            # DRAW BBOX
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # CENTER POIN
            cv2.circle(frame, (cx, cy), 4, color, -1)

            # MOTION VECTOR
            if state == "moving":
                arrow_scale = 3
                end_point = (
                    int(cx + dx * arrow_scale),
                    int(cy + dy * arrow_scale)
                )
                cv2.arrowedLine(
                    frame,
                    (cx, cy),
                    end_point,
                    (0, 255, 255),
                    3,
                    tipLength=0.4
                )

            # LABEL
            cv2.putText(
                frame,
                f"ID {track_id} | {state} | v={speed:.1f}px",
                (x1, y1 - 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

    # ================== COUNTERS ==================
    cv2.putText(frame, f"People: {len(current_ids)}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    cv2.putText(frame, f"All detected: {len(seen_track_ids)}", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 255), 2)

    cv2.putText(frame, f"Frame: {frame_id}", (10, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.imshow("YOLO + Tracking + Motion Visualization", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        running = False

# CLEANUP
cap.release()
cv2.destroyAllWindows()
print(f"Завершено. Всего уникальных fact detect: {len(seen_track_ids)}")


Загрузка модели...
▶️ Старт обработки. Нажми Q или Ctrl+C
Завершено. Всего уникальных fact detect: 0


In [12]:
# Глобальная переменная для graceful shutdown
running = True

def signal_handler(sig, frame):
    """Обработчик сигнала Ctrl+C"""
    global running
    print("\nПолучен сигнал остановки...")
    running = False

# Регистрация обработчика сигнала
signal.signal(signal.SIGINT, signal_handler)

def dice_coef(gt_mask, pred_mask):
    """
    Коэффициент Дайса для сравнения масок
    """
    gt = gt_mask.astype(bool)
    pr = pred_mask.astype(bool)
    inter = (gt & pr).sum()
    return 2 * inter / (gt.sum() + pr.sum() + 1e-6)

def process_video_with_tracking(model, source, tracker='bytetrack.yaml', conf=0.5, show=True, save=False):
    global running
    
    # Открыть источник видео (видеофайл или поток)
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise Exception(f"Ошибка: Не удалось открыть источник: {source}")
    
    # Получить параметры видео
    fps = int(cap.get(cv2.CAP_PROP_FPS)) if source.endswith(('.mp4', '.avi', '.mov')) else 30
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if save:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        output_path = "output_video.mp4"
        out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    frame_count = 0
    print(f"Начало обработки. Нажмите 'q' в окне видео или Ctrl+C в консоли для остановки.")

    while running:
        ret, frame = cap.read()

        if not ret:
            print("Не удалось получить кадр. Повторная попытка...")
            cv2.waitKey(1000)
            continue
        
        frame_count += 1
        
        try:
            print(f"Обрабатываем кадр {frame_count}")
            
            # Преобразуем conf в обычный float
            conf = float(conf)
            
            results = model.track(
                frame,
                classes=[0],  
                iou=0.4,      
                conf=conf,     
                persist=True,  
                imgsz=640,     
                verbose=False, 
                tracker=tracker    
            )

            annotated_frame = frame.copy()
            people_count = 0

            if len(results) > 0 and results[0].boxes is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                confidences = results[0].boxes.conf.cpu().numpy()
                people_count = len(boxes)
                
                for i, (box, conf) in enumerate(zip(boxes, confidences)):
                    x1, y1, x2, y2 = map(int, box)
                    color = (0, 255, 0)
                    cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
                    label = f'Человек {conf:.2f}'
                    cv2.putText(annotated_frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            cv2.putText(annotated_frame, f'Shot: {frame_count}', (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.putText(annotated_frame, f'Peoples: {people_count}', (10, 60),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            if save:
                out.write(annotated_frame)

            if show:
                display_frame = cv2.resize(annotated_frame, (0, 0), fx=0.75, fy=0.75)
                cv2.imshow("YOLOv — Обнаружение людей", display_frame)
                key = cv2.waitKey(1) & 0xFF
                if key == ord("q"):
                    running = False
                elif key == 27:  # ESC
                    running = False

        except Exception as e:
            print(f"Ошибка при обработке кадра {frame_count}: {e}")
            continue

    cap.release()
    if save:
        out.release()

    cv2.destroyAllWindows()
    print(f"Обработка завершена. Обработано кадров: {frame_count}")



    # Освобождение ресурсов
    cap.release()
    if save:
        out.release()

    # Закрытие всех окон OpenCV
    cv2.destroyAllWindows()
    print(f"Обработка завершена. Обработано кадров: {frame_count}")

# Пример использования:
if __name__ == "__main__":
    # Загрузка модели
    print("Загрузка модели...")
    model = YOLO('runs/result/preprocessdetect2.pt')

    # Для видеопотока
    source = "https://restreamer.vms.evo73.ru/918335436b92ac26/stream.m3u8  "

    try:
        # Запуск обработки видео
        process_video_with_tracking(
            model,
            source=source.strip(),
            tracker='bytetrack.yaml',
            conf=0.5,
            show=True,
            save=False
        )
    except Exception as e:
        print(f"Ошибка: {e}")
    finally:
        print("Программа завершена.")

Загрузка модели...


FileNotFoundError: [Errno 2] No such file or directory: 'runs\\result\\preprocessdetect2.pt'

In [ ]:
# GRACEFUL SHUTDOWN
running = True

def signal_handler(sig, frame):
    global running
    print("\nОстановка...")
    running = False

signal.signal(signal.SIGINT, signal_handler)

# MODEL
print("Загрузка модели...")
model = YOLO("runs/result/preprocessdetect2.pt")

# VIDEO SOURCE (ССЫЛКА!)
SOURCE = "https://video2.interra.ru/glaz.naroda.121-c9e033c05f/index.m3u8?token=3.9CzUU5u-AAAAAAAAAEsAAAAAAAAAAHTKPf595bILj3OTg5Az34JUkT1F"

cap = cv2.VideoCapture(SOURCE)
if not cap.isOpened():
    raise RuntimeError("Не удалось открыть видеопоток")

# =========================
# TRACK STORAGE
# =========================
tracks = defaultdict(list)

print("Старт обработки. Нажми Q или Ctrl+C")

frame_id = 0

while running and cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Потеря кадра, повтор...")
        cv2.waitKey(500)
        continue

    frame_id += 1

    # =========================
    # YOLO + TRACKING
    # =========================
    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.4,
        iou=0.4,
        classes=[0],  # человек
        verbose=False
    )

    if results and results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy()

        for box, track_id in zip(boxes, ids):
            x1, y1, x2, y2 = map(int, box)

            # центр bbox
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            tracks[track_id].append((cx, cy))

            # =========================
            # BEHAVIOR ANALYSIS
            # =========================
            state = "unknown"

            if len(tracks[track_id]) >= 2:
                dx = tracks[track_id][-1][0] - tracks[track_id][-2][0]
                dy = tracks[track_id][-1][1] - tracks[track_id][-2][1]
                speed = np.sqrt(dx*dx + dy*dy)

                if speed < 2:
                    state = "standing"
                    color = (0, 0, 255)
                else:
                    state = "moving"
                    color = (0, 255, 0)
            else:
                color = (255, 255, 0)

            # =========================
            # DRAW
            # =========================
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                frame,
                f"ID {int(track_id)} | {state}",
                (x1, y1 - 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

    cv2.putText(
        frame,
        f"Frame: {frame_id}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    cv2.imshow("YOLO + Tracking + Behavior", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        running = False

# CLEANUP
cap.release()
cv2.destroyAllWindows()
print("Завершено")


Загрузка модели...
Старт обработки. Нажми Q или Ctrl+C
Завершено
